# RTVIO benchmark on MARS-LVIG HKairport_GNSS03 (RTK ground truth)

Converts the bag to an RTVIO session once, runs the reconstruction configurations below, and scores every run's camera trajectory against the drone's RTK:

- **absolute** error of the georeferenced cameras (horizontal / vertical, share within the 1 m target) - what the problem statement asks for;
- **shape** error after a best-fit Sim(3) - how good the trajectory is regardless of GPS;
- frames/s, peak VRAM, and fallback / GPS-scaled seam counts from each run's report.

The pipeline only ever sees the **standalone u-blox** GPS (no RTK corrections); the RTK track is ground truth only. Converter details (18 s GPS-UTC shift, altitude-reference offset, takeoff/landing trim) are in `tools/marslvig_to_session.py`; it verifies every clock against the bag's receive time and prints the result.

**Before running**
1. Accelerator **GPU T4 x2**, **Internet: On**.
2. **Add Input -> Your Work**: the `mars-lvig-hkairport-gnss03` dataset and the `vggt1b-rtvio` model (no 9 GB / 5 GB re-download each session).

In [ ]:
import glob, json, os, subprocess, sys, time
import torch

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

REPO_DIR = "/kaggle/working/Terraform"
if not os.path.exists(REPO_DIR):
    !git clone --branch linux-dev --recurse-submodules https://github.com/Jola7898/Terraform.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q
!cd {REPO_DIR} && git log --oneline -1
RTVIO = os.path.join(REPO_DIR, "rtvio")

# huggingface_hub deliberately not upgraded (see kaggle_fetch_assets.ipynb).
!pip install -q -e {RTVIO} rosbags einops safetensors laspy
!cd {RTVIO} && python tests/test_marslvig_tools.py | tail -1 && python tests/test_gps_seams.py | tail -1

In [ ]:
def find_one(pattern, what):
    hits = glob.glob(pattern, recursive=True)
    assert hits, "%s not found under /kaggle/input - attach it via Add Input -> Your Work" % what
    return hits[0]

BAG = find_one("/kaggle/input/**/HKairport_GNSS03.bag", "the mars-lvig-hkairport-gnss03 dataset")
CKPT = find_one("/kaggle/input/**/vggt1b_model.pt", "the vggt1b-rtvio model")
CALIB = os.path.join(RTVIO, "data", "marslvig", "HK_GNSS_airport_island.yaml")
os.environ["RTVIO_VGGT_CHECKPOINT"] = CKPT
print("bag  :", BAG, "\nckpt :", CKPT, "\ncalib:", CALIB)

## 1. Bag -> RTVIO session (once per session; ~2-4 min)

`STRIDE = 2` keeps 5 of the camera's 10 frames/s - still heavy overlap at airport-flight altitude, and half the GPU time. Check the printed `clock_vs_bag_receive_time` (every median should be ~0 s) and `datum` (the RTK - u-blox altitude offset the scorer removes).

In [ ]:
STRIDE = 2
SESSION = "/tmp/hkairport_gnss03_s%d" % STRIDE

if not os.path.exists(os.path.join(SESSION, "session_meta.json")):
    subprocess.run([sys.executable, "tools/marslvig_to_session.py", "--bag", BAG, "--calib", CALIB,
                    "--out", SESSION, "--stride", str(STRIDE)], check=True, cwd=RTVIO)
else:
    print(json.dumps(json.load(open(os.path.join(SESSION, "session_meta.json"))), indent=2))

## 2. Reconstruction runs

`global` = the paper's behaviour (one GPS fit after fusion); `guided` = GPS also checks each window seam. The 1-GPU pair isolates the GPS change; the 2-GPU run measures the throughput gain on otherwise identical settings. Meshing is off - this benchmark scores trajectories. Each run's full log is saved next to its outputs; runs already finished are skipped on re-execution.

In [ ]:
RUNS = [
    dict(name="vggt_global_1gpu", gps_mode="global", gpus=1),
    dict(name="vggt_guided_1gpu", gps_mode="guided", gpus=1),
    dict(name="vggt_guided_2gpu", gps_mode="guided", gpus=2),
]
BENCH = "/kaggle/working/bench_hkairport_gnss03_s%d" % STRIDE

for run in RUNS:
    out = os.path.join(BENCH, run["name"])
    if os.path.exists(os.path.join(out, "CHECKPOINT_REPORT.json")):
        print("skip (done):", run["name"])
        continue
    os.makedirs(out, exist_ok=True)
    cmd = [sys.executable, "-u", "-m", "rtvio.vggt_reconstruct", "--from-recording", SESSION, "--out", out,
           "--gps-mode", run["gps_mode"], "--gpus", str(run["gpus"]), "--no-mesh"]
    print("\n=== %s: %s" % (run["name"], " ".join(cmd[2:])))
    t = time.time()
    with open(os.path.join(out, "run.log"), "w") as log:
        p = subprocess.run(cmd, cwd=RTVIO, stdout=log, stderr=subprocess.STDOUT)
    print("exit %d after %.1f min; last lines:" % (p.returncode, (time.time() - t) / 60))
    !tail -n 12 {os.path.join(out, "run.log")}

## 3. Score against RTK

Paste this table back. Caveats: the camera-to-RTK-antenna lever arm is not corrected (tens of cm), and ground truth is RTK shifted by the constant altitude-reference offset the converter measured (horizontal untouched).

In [ ]:
done = [os.path.join(BENCH, r["name"]) for r in RUNS
        if os.path.exists(os.path.join(BENCH, r["name"], "cameras.json"))]
cmd = [sys.executable, "tools/eval_trajectory.py", "--session", SESSION] + sum([["--run", d] for d in done], [])
print(subprocess.run(cmd, cwd=RTVIO, capture_output=True, text=True, check=True).stdout)